In [1]:
# Cell 0 — Load Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Load datasets
marketing_ads   = pd.read_csv('marketing_ads_data_clean.csv')
sales_txn       = pd.read_csv('sales_transaction_clean.csv')
ad_summary      = pd.read_csv('ad_summary_clean.csv')

# Preview
print("=== Marketing Ads Data ===")
print(f"Shape: {marketing_ads.shape}")
print(f"Columns: {list(marketing_ads.columns)}")
print("\nPreview 5 baris pertama:")
print(marketing_ads.head(5).to_string())

print("\n=== Sales_txn ===")
print(f"Shape: {sales_txn.shape}")
print(f"Columns: {list(sales_txn.columns)}")
print("\nPreview 5 baris pertama:")
print(sales_txn.head(5).to_string())

print("\n=== ad_summary ===")
print(f"Shape: {ad_summary.shape}")
print(f"Columns: {list(ad_summary.columns)}")
print("\nPreview 5 baris pertama:")
print(ad_summary.head(5).to_string())

=== Marketing Ads Data ===
Shape: (1000, 12)
Columns: ['ad_id', 'campaign_name', 'audience_type', 'product_type', 'product_name', 'variant', 'date', 'impressions', 'clicks', 'spend_idr', 'sessions', 'add_to_cart']

Preview 5 baris pertama:
     ad_id         campaign_name audience_type product_type                   product_name         variant        date  impressions  clicks  spend_idr  sessions  add_to_cart
0  AD_0032    Awareness Campaign          cold        Serum    DermaGlow Brightening Serum    Video_Review  2023-01-06         8060     253   458896.0       216           29
1  AD_0110    Awareness Campaign          cold    Sunscreen  DermaGlow UV Shield Sunscreen    Video_Review  2023-09-01        17314     542   108070.0       452           72
2  AD_0137  Retargeting Campaign          warm    Sunscreen  DermaGlow UV Shield Sunscreen  Static_Catalog  2023-04-13        16374     180   135435.0       146            8
3  AD_0089  Retargeting Campaign          warm    Sunscreen  Der

In [2]:
# CELL 1 - Build Funnel Table per Variant

# 1. Funnel atas dari marketing_ads
# Source: marketing_ads
# Stage: Impressions - Clicks - Session - Add to Cart

funnel_ads = marketing_ads.groupby('variant').agg(
    impressions = ('impressions', 'sum'),
    clicks      = ('clicks', 'sum'),
    sessions    = ('sessions', 'sum'),
    add_to_cart = ('add_to_cart', 'sum')
).reset_index()

# 2. Funnel bawah dari sales_txt
# Source: sales_transaction_clean
# Stage: Transaction - Purchase

funnel_sales = sales_txn.groupby('variant').agg(
    transactions = ('transaction_id', 'count'),
    revenue_idr = ('revenue_idr', 'sum')
    ).reset_index()

# 3. Merge kedua funnel
funnel = funnel_ads.merge(funnel_sales, on='variant')

print("=== Funnel Table per Variant ===")
print(funnel.to_string(index=False))

=== Funnel Table per Variant ===
       variant  impressions  clicks  sessions  add_to_cart  transactions  revenue_idr
Static_Catalog      6077373   72240     61177         7282          1200  343045000.0
  Video_Review      6223089  155948    132353        16241           731  207661000.0


In [ ]:
# CELL 2 - Drop Off Calculation Analysis

# Drop off rate = berapa persen yang tidak lanjut ke stage berikutnya
# Formula: (stage sekarang - stage berikutnya) / stage sekarang * 100

stages = ['impressions', 'clicks', 'sessions', 'add_to_cart', 'transactions']
transitions = [
    ('impressions', 'clicks'),
    ('clicks', 'sessions'),
    ('sessions', 'add_to_cart'),
    ('add_to_cart', 'transactions')
]

records = []
for _, row in funnel.iterrows():
    for from_stage, to_stage in transitions:
        dropoff_rate = (1-row[to_stage] / row[from_stage]) * 100
        conversion_rate = (row[to_stage] / row[from_stage]) * 100
        records.append({
            'variant': row['variant'],
            'from_stage': from_stage,
            'to_stage': to_stage,
            'from_value': int(row[from_stage]),
            'to_value': int(row[to_stage]),
            'conversion_rate': round(conversion_rate, 2),
            'dropoff_rate': round(dropoff_rate, 2)
        })
dropoff_df = pd.DataFrame(records)

print("=== Drop Off Rates per Stage per Variant ===")
print(dropoff_df.to_string(index=False))

=== Drop Off Rates per Stage per Variant ===
       variant  from_stage     to_stage  from_value  to_value  conversion_rate  dropoff_rate
Static_Catalog impressions       clicks     6077373     72240             1.19         98.81
Static_Catalog      clicks     sessions       72240     61177            84.69         15.31
Static_Catalog    sessions  add_to_cart       61177      7282            11.90         88.10
Static_Catalog add_to_cart transactions        7282      1200            16.48         83.52
  Video_Review impressions       clicks     6223089    155948             2.51         97.49
  Video_Review      clicks     sessions      155948    132353            84.87         15.13
  Video_Review    sessions  add_to_cart      132353     16241            12.27         87.73
  Video_Review add_to_cart transactions       16241       731             4.50         95.50


# CELL 2 (Drop Off Calculation Analysis) - INSIGHT AND KEY FINDINGS

### Yang saya lakukan:
Menghitung drop off rate di setiap transisi funnel (impressions - clicks - sessions - add to cart - transactions) pada
kedua variant untuk mengetahui dimana kebocoran terbesar terjadi. 

### Key Findings:
- **Finding 1:** Drop Off terbesar kedua variant terjadi pada stage ***Impressions - Clicks*** (Static Catalog 98% dan Video Review 97%), namun Video Review memiliki CTR 2x lebih tinggi (2.51% vs 1.19%) menunjukkan bahwa Vudeo Review lebih efektif untuk menarik perhatian audience. 
- **Finding 2:** Pada stage ***Clicks - Sessions***, kedua variant menunjukkan angka Drop Off Rate yang hampir mirip (Static Catalog 15.31% dan Video Review 15.13%), artinya kualitas landing page/website bukan menjadi masalah utama.
- **FInding 3:** Drop Off paling signifikan terjadi pada stage ***Add to Cart - Transactions***, pada stage ini Static Catalog (drop off rate 83%, Conv Rate 16.48%), Video Review (drop off 95%, Conv Rate 4.50%). Static Catalog (7.282 atc - 1.200 transaksi) sementara Video Review (16.241 atc - 731 transaksi), mengindikasikan Static Catalog sangat efektif konversi audience membeli produk. 
### Insight:
Video Review efektif untuk funnel, menarik banyak klik namun tingginya dropp off di tahap akhir mengindikasikan audience tertarik melihat video namun belum memiliki purchase intent. Sementara Static Catalog efektif pada funnel bawah, menarik sedikit audience namun setiap audience yang masuk ke funnel cenderung purchase ready. 

### Business Implication: 
Apabila tujuan campaign adalah konversi maka Static Catalog prioritas utama, apabila orientasi campaign adalah Top of Funnel maka Video Review bisa menjadi alat untuk menarik audience dengan memperkenalkan produk sebelum audience di retarget dengan Static Catalog. 



In [ ]:
# CELL 3 - Funnel Quality per Segment Analysis

# Objektif: Brekdown drop off rate Add to Cart - Transactions per Variant + Audience Type, per Variant + Product Type
# Source: marketing_ads (Funnel atas) + sales_txn (Funnel bawah)

# Enrich sales_txn dengan audience_type & product_type sales_txn tidak punya kolom audience_type & product_type, jadi kita join dulu ke marketing_ads untuk dapat attribute per ad_id

txn_enriched = sales_txn.merge(
    marketing_ads[['ad_id', 'audience_type', 'product_type']].drop_duplicates(subset='ad_id'),
    on='ad_id',
    how='left'
)

# Segment 1: per Variant + Audience Type

# Funnel atas: Add to Cart per variant + audience type
ads_by_audience = marketing_ads.groupby(['variant', 'audience_type']).agg(
    impressions = ('impressions', 'sum'),
    clicks      = ('clicks', 'sum'),
    sessions    = ('sessions', 'sum'),
    add_to_cart = ('add_to_cart', 'sum')
).reset_index()

# Funnel bawah: Transactions per variant + audience type
txn_by_audience     = txn_enriched.groupby(['variant', 'audience_type']).agg(
    transactions    = ('transaction_id', 'count'),
    revenue_idr     = ('revenue_idr', 'sum')
).reset_index()

# Merge Funnel atas dan bawah
segment_audience = ads_by_audience.merge(txn_by_audience, on=['variant', 'audience_type'])

# Hitung conversion dan drop off tiap stage
segment_audience['CTR']                 = (segment_audience['clicks'] / segment_audience['impressions'] * 100).round(2)
segment_audience['sessions_rate']       = (segment_audience['sessions'] / segment_audience['clicks'] * 100).round(2)
segment_audience['atc_rate']            = (segment_audience['add_to_cart'] / segment_audience['sessions'] * 100).round(2)
segment_audience['cart_conv_rate']      = (segment_audience['transactions'] / segment_audience['add_to_cart'] * 100).round(2) 
segment_audience['overall_conv_rate']   = (segment_audience['transactions'] / segment_audience['impressions'] * 100).round(2)

print("=== Funnel Quality per Variant + Audience Type ===")
print(segment_audience.to_string(index=False))

# Segment 2: per Variant + Product Type
ads_by_product  = marketing_ads.groupby(['variant', 'product_type']).agg(
    impressions = ('impressions', 'sum'),
    clicks      = ('clicks', 'sum'),
    sessions    = ('sessions', 'sum'),
    add_to_cart = ('add_to_cart', 'sum')
).reset_index()

txn_by_product = sales_txn.groupby(['variant', 'product_type']).agg(
    transactions = ('transaction_id', 'count'),
    revenue_idr = ('revenue_idr', 'sum')
).reset_index()

segment_product = ads_by_product.merge(txn_by_product, on=['variant', 'product_type'])

segment_product['CTR'] = (segment_product['clicks'] / segment_product['impressions'] * 100).round(2)
segment_product['session_rate'] = (segment_product['sessions'] / segment_product['clicks'] * 100).round(2)
segment_product['atc_rate'] = (segment_product['add_to_cart'] / segment_product['sessions'] * 100).round(2)
segment_product['cart_conv_rate'] = (segment_product['transactions'] / segment_product['add_to_cart'] * 100).round(2)
segment_product['overall_conv_rate'] = (segment_product['transactions'] / segment_product['impressions'] * 100).round(4)

print("\n=== Funnel Quality per Variant + Product Type ===")
print(segment_product.to_string(index=False))

=== Funnel Quality per Variant + Audience Type ===
       variant audience_type  impressions  clicks  sessions  add_to_cart  transactions  revenue_idr  CTR  sessions_rate  atc_rate  cart_conv_rate  overall_conv_rate
Static_Catalog          cold      3995888   47700     40397         4824           726  203494000.0 1.19          84.69     11.94           15.05               0.02
Static_Catalog          warm      2081485   24540     20780         2458           441  128818000.0 1.18          84.68     11.83           17.94               0.02
  Video_Review          cold      4178288  104547     88620        10916           461  129654000.0 2.50          84.77     12.32            4.22               0.01
  Video_Review          warm      2044801   51401     43733         5325           233   66916000.0 2.51          85.08     12.18            4.38               0.01

=== Funnel Quality per Variant + Product Type ===
       variant product_type  impressions  clicks  sessions  add_to_cart  

# CELL 3 (Funnel Quality per Segment) - INSIGHT AND KEY FINDINGS

## Yang saya lakukan: 
Melakuan analisis kualitas funnel per segment dengan breakdown stage ***Add to Cart to Transactions*** kedua variant dengan subjek Audience Type dan Product Type untuk mengetahui segment konsumen yang lanjut transkaksi produk setelah Add to Cart. 

### Key Findings:
- **Finding 1:** Static Catalog unggul pada Cart Conversion di kedua segment audience, Cold (15.05% vs 4.22) dan Warm (17.94% vs 4.38%). Gap antar variant lebih signifikan dibanding gap antar audience type, mengindikasikan performa bukan karena target audience melainkan kualitas creative pada Video Review.
- **Finding 2:** Pola yang sama terlihat pada semua Product Type, Static Catalog selalu unggul di Cart Conversion (15%-17%) vs Video Review (3%-5%) pada setiap kategori produk. Sementara Video Review konsisten unggul pada CTR (2.47-2.56% vs 1.17-1.21%), mengindikasikan Static Catalog efektif pada bottom funnel dan Video Review efektif pada top funnel.  
- **Finding #:** Produk Sunscreen menjadi produk paling perform pada kedua variant, Video review (CTR 2.56%) dan Static Catalog (Cart Conversion Rate 17.20%). Sunscreen menjadi prioritas dalam campaign apapun kedepannya. 

### Insight:
Static Catalog selalu unggul di setiap audience type dan semua product type, mengindikasikan format catalog yang menampilkan spesifikasi produk, tampilan produk dan harga langsung lebih mendorong purchase intent dari audience audience type itu sendiri. Video Review banyak yang klik tapi belum purhcase ready.  

### Business Implication:
Retargeting Cold dan Warm audience yang sudah melihat dan mengklik video menggunakan Static Catalog untuk melakukan pembelian karena Static Catalog efektif pada bottom funnel. Pertimbangkan juga evaluasi pada kualitas Video Review seperti copywrting untuk meningkatkan konversi pada variant Video Review bukan hanya click through.  

In [24]:
# CELL 4 - Business Impact Estimation

# Objektif: Estimasi revenue yang hilang akibat drop off tinggi dan potensi revenue tambahan jika cart conversion rate bisa diperbaiki
# Fokus: stage Add to Cart - Transactions (drop off plaing krusial)

# Baseline metrics
avg_order_value = sales_txn.groupby('variant')['revenue_idr'].mean().round(0)
print("=== Average Order Value per Variant ===")
print(avg_order_value)

# Skenario: Bagaimana jika Video_Review memiliki cart conversion rate yang sama dengan Static_Catalog?
static_cart_conv = funnel.loc[funnel['variant'] == 'Static_Catalog', 'transactions'].values[0] / funnel.loc[funnel['variant'] == 'Static_Catalog', 'add_to_cart'].values[0]

video_atc = funnel.loc[funnel['variant'] == 'Video_Review', 'add_to_cart'].values[0]
video_actual_txn = funnel.loc[funnel['variant'] == 'Video_Review', 'transactions'].values[0]
video_aov = avg_order_value['Video_Review']

# Transaksi yang bisa didapat jika Video_Review punya cart conversion rate yang sama dengan Static_Catalog
video_potential_txn = round(video_atc * static_cart_conv)
video_additional_txn = video_potential_txn - video_actual_txn
video_lost_revenue = round(video_additional_txn * video_aov)

print("\n=== Business Impact: Video Review cart conversion gap ===")
print(f"static catalog cart conv rate: {static_cart_conv:.2%}")
print(f"video review add to cart: {int(video_atc):,}")
print(f"video review actual transactions: {int(video_actual_txn):,}")
print(f"video review potential trans: {int(video_potential_txn):,}")
print(f"additional transactions unrealized: {int(video_additional_txn):,}")
print(f"estimated lost revenue: Rp {video_lost_revenue:,.0f}")

# Skenario 2: Apabila drop off impressions - clicks Static Catalog ditingkatkan ke level Video Review
static_impressions = funnel.loc[funnel['variant'] == 'Static_Catalog', 'impressions'].values[0]
video_ctr = funnel.loc[funnel['variant'] == 'Video_Review', 'clicks'].values[0] / funnel.loc[funnel['variant'] == 'Video_Review', 'impressions'].values[0]
static_actual_clicks = funnel.loc[funnel['variant'] == 'Static_Catalog', 'clicks'].values[0]
static_session_rate = funnel.loc[funnel['variant'] == 'Static_Catalog', 'sessions'].values[0] / funnel.loc[funnel['variant'] == 'Static_Catalog', 'clicks'].values[0]
static_atc_rate = funnel.loc[funnel['variant'] == 'Static_Catalog', 'add_to_cart'].values[0] / funnel.loc[funnel['variant'] == 'Static_Catalog', 'sessions'].values[0]
static_aov = avg_order_value['Static_Catalog']

# Simulasi apabila CTR Static Catalog setara Video Review
static_potential_clicks = round(static_impressions * video_ctr)
static_potential_sessions = round(static_potential_clicks * static_session_rate)
static_potential_atc = round(static_potential_sessions * static_atc_rate)
static_potential_txn = round(static_potential_atc * static_cart_conv)
static_actual_txn = funnel.loc[funnel['variant'] == 'Static_Catalog', 'transactions'].values[0]
static_additional_txn = static_potential_txn - static_actual_txn
static_additional_rvn = round(static_additional_txn * static_aov)

print("\n=== Business Impact: Static Catalog CTR Improvement Scenario ===")
print(f"video review CTR: {video_ctr:.2%}")
print(f"static catalog actual clicks: {int(static_actual_clicks):,}") 
print(f"static catalog potential clicks: {int(static_potential_clicks):,}") 
print(f"static catalog potential transactions: {int(static_potential_txn):,}")
print(f"static_additional transactions: {int(static_additional_txn):,}")
print(f"Estimation additional revenue: Rp {int(static_additional_rvn):,.0f}")

=== Average Order Value per Variant ===
variant
Static_Catalog    285871.0
Video_Review      284078.0
Name: revenue_idr, dtype: float64

=== Business Impact: Video Review cart conversion gap ===
static catalog cart conv rate: 16.48%
video review add to cart: 16,241
video review actual transactions: 731
video review potential trans: 2,676
additional transactions unrealized: 1,945
estimated lost revenue: Rp 552,531,710

=== Business Impact: Static Catalog CTR Improvement Scenario ===
video review CTR: 2.51%
static catalog actual clicks: 72,240
static catalog potential clicks: 152,296
static catalog potential transactions: 2,530
static_additional transactions: 1,330
Estimation additional revenue: Rp 380,208,430


# CELL 4 (Business Impact Estimation)

## Yang saya lakukan:
Melakukan estimasi revenue yang hilang akibat drop off pada stage Add to Cart - Transactions dan potensi revenue tambahan dari perbaikan CTR menggunakan Average Order Value per Variant sebagai basis kalkulasi (Static Catalog 285.871 & Video Review 284.078).

## Key Findings:
- **Finding 1:** Video Review mendapatkan 16.241 atc namun yang ter realisasi hanya 731 transaksi. Apabila conversion rate Video Review menyamai Static Catalog (16.48%) maka potensi transaksi yang dapat terjadi adalah 2.676, maka terdapat 1.945 transaksi sia sia setara dengan kehilangan revenue sebesar Rp. 552.531.710. 
- **Finding 2:** Static Catalog mendapatkan 72.240 klik, apabila CTR Static Catalog menyamai Video Review (2,51%), maka potensi tambahan transaksi yang bisa didapatkan adalah 1.330 transaksi atau setara Rp. 380.208.430. 

## Insight: 
Skenario 1 menjadi prioritas utama karena mampu menarik audience secara masif namun bocor pada tahap akhir (Transaksi). Apabila di perbaiki, maka potensi revenue Video Review adalah Rp. 552.531.710 tanpa perlu menambah budget untuk iklan. Sementara Skenario 2 butuh investasi lebih untuk menarik traffic baru namun dengan return yang lebih kecil Rp. 380.208.430. 




# PROJECT SUMMARY

- **Objective:** Menganalisis Funnel Marketing brand skincare fiktif DermaGlow untuk mengidentifikasi drop off terbesar terjadi dan mengukur business impact sebagai lanjutan dari A/B Testing Analysis (Video Review vs Static Catalog). 

- **Business Question:** 
    1. Di stage mana kebocoran funnel terbesar terjadi, dan variant mana 
    yang lebih efisien?
    2. Apakah perbedaan performa antar variant konsisten di semua segment 
    (audience type & product type)?
    3. Berapa estimasi revenue yang hilang akibat drop-off, dan berapa 
    potensi revenue jika drop-off diperbaiki?

- **Dataset:** 
    1. Marketing Ads Data
    2. Sales Transactions Data
    3. Ad Summary (Gabungan data dari Marketing dan Sales)

- **Key Findings:**
    1. Drop-off kritis ada di Add to Cart → Transactions, bukan di stage lain
    2. Static_Catalog konsisten unggul di cart conversion rate di semua segment (~4x Video_Review)
    3. Video_Review efektif top funnel (CTR 2x lebih tinggi) tapi bocor di bottom funnel
    4. Sunscreen adalah hero product — top performer di kedua variant
    5. 1.945 transaksi Video_Review tidak terealisasi = Rp552 juta revenue hilang

- **Tools Used:**
    1. **Python** (pandas, numpy, scipy) — data cleaning, analysis, statistical testing
    2. **Jupyter Notebook** — interactive analysis environment
    3. **Power BI** — dashboard & visualization
    4. **GitHub** — version control & portfolio documentation

In [ ]:
# CELL 5 - Setup directory untuk Power BI Export
import os
os.makedirs('output', exist_ok=True)

In [ ]:
# CELL 6 - Export Data untuk Power BI

# Menambahkan kolom label untuk Power BI readability

# Funnel table dengan label stage (untuk waterfall / funnel chart di Power BI)
funnel_long = pd.melt(
    funnel[['variant', 'impressions', 'clicks', 'sessions', 'add_to_cart', 'transactions']],
    id_vars='variant',
    var_name='stage',
    value_name='count'
)

# Membuat urutan stage agar lebih rapih dan Power BI bisa sort dengan benar
stage_order = {
    'impressions' : 1,
    'clicks'      : 2,
    'sessions'    : 3,
    'add_to_cart' : 4,
    'transactions': 5
}
funnel_long['stage_order'] = funnel_long['stage'].map(stage_order)
funnel_long = funnel_long.sort_values(['variant', 'stage_order'])

# Tambah dropoff_rate ke dropoff_df 
# Sudah ada dari Cell 2, tinggal export

# Business impact summary table 
impact_summary = pd.DataFrame([
    {
        'scenario'                  : 'Fix Video_Review Cart Conversion',
        'variant'                   : 'Video_Review',
        'actual_transactions'       : video_actual_txn,
        'potential_transactions'    : video_potential_txn,
        'additional_transactions'   : video_additional_txn,
        'estimated_impact_idr'      : video_lost_revenue,
        'description'               : 'Revenue lost due to low cart conversion vs Static_Catalog'
    },
    {
        'scenario'                  : 'Boost Static_Catalog CTR',
        'variant'                   : 'Static_Catalog',
        'actual_transactions'       : static_actual_txn,
        'potential_transactions'    : static_potential_txn,
        'additional_transactions'   : static_additional_txn,
        'estimated_impact_idr'      : static_additional_rvn,
        'description'               : 'Revenue upside if Static_Catalog matches Video_Review CTR'
    }
])

# Export CSV
funnel_long.to_csv('output/funnel_by_variant.csv', index=False)
dropoff_df.to_csv('output/dropoff_rates.csv', index=False)
segment_audience.to_csv('output/segment_by_audience.csv', index=False)
segment_product.to_csv('output/segment_by_product.csv', index=False)
impact_summary.to_csv('output/business_impact.csv', index=False)

print("=== Export Summary ===")
print(f"funnel_by_variant.csv   → {funnel_long.shape[0]} rows, {funnel_long.shape[1]} cols")
print(f"dropoff_rates.csv       → {dropoff_df.shape[0]} rows, {dropoff_df.shape[1]} cols")
print(f"segment_by_audience.csv → {segment_audience.shape[0]} rows, {segment_audience.shape[1]} cols")
print(f"segment_by_product.csv  → {segment_product.shape[0]} rows, {segment_product.shape[1]} cols")
print(f"business_impact.csv     → {impact_summary.shape[0]} rows, {impact_summary.shape[1]} cols")
print("\n✅ Semua file berhasil di-export ke folder output/")

=== Export Summary ===
funnel_by_variant.csv   → 10 rows, 4 cols
dropoff_rates.csv       → 8 rows, 7 cols
segment_by_audience.csv → 4 rows, 13 cols
segment_by_product.csv  → 8 rows, 13 cols
business_impact.csv     → 2 rows, 7 cols

✅ Semua file berhasil di-export ke folder output/
